# CelebA Face Reconstruction with a Convolutional Autoencoder

## Project Overview

This notebook implements a **convolutional autoencoder** that learns to compress
and reconstruct face images from the [CelebA](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html)
dataset (Large-scale CelebFaces Attributes Dataset).

The model is trained in a fully unsupervised fashion: it learns a compact latent
representation of a face and then reconstructs the original image from that
representation, using pixel-wise reconstruction error (MSE) as the training
signal.

**What this notebook covers:**
- Loading and preprocessing a subset of the CelebA dataset
- Defining a convolutional encoder-decoder architecture in PyTorch
- Training the autoencoder and tracking train/validation loss
- Visualizing reconstruction quality against the original images

**Tech stack:** Python, PyTorch, Torchvision, Matplotlib


In [1]:
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

# Avoids a known OpenMP duplicate-runtime conflict between PyTorch and other
# scientific libraries on some systems (mainly Windows/macOS).
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


## Configuration

All hyperparameters and paths are centralized in a single `Config` dataclass.
This keeps the experiment reproducible and makes it trivial to tweak settings
without hunting through the notebook for magic numbers.


In [2]:
@dataclass
class Config:
    '''Hyperparameters and settings for the CelebA autoencoder experiment.

    Attributes:
        batch_size: Number of samples per training/validation batch.
        epochs: Number of full passes over the training data.
        learning_rate: Step size used by the Adam optimizer.
        image_size: Height and width (in pixels) images are resized to.
        channels: Number of image channels (3 for RGB).
        train_subset_size: Number of training images sampled from CelebA.
        val_subset_size: Number of validation images sampled from CelebA.
        data_root: Directory containing the pre-downloaded CelebA face
            images (a folder of ``.jpg`` files, e.g. ``img_align_celeba``).
        seed: Random seed used for reproducibility.
    '''

    batch_size: int = 64
    epochs: int = 100
    learning_rate: float = 1e-4
    image_size: int = 64
    channels: int = 3
    train_subset_size: int = 5000
    val_subset_size: int = 500
    data_root: str = r"D:\UNI AI\Auto_Encoder\data\celeba\img_align_celeba"
    seed: int = 42


CONFIG = Config()
CONFIG


Config(batch_size=64, epochs=100, learning_rate=0.0001, image_size=64, channels=3, train_subset_size=5000, val_subset_size=500, data_root='D:\\UNI AI\\Auto_Encoder\\data\\celeba\\img_align_celeba', seed=42)

## Reproducibility and Device Setup

We seed every relevant random number generator so that dataset subsetting and
model initialization are reproducible across runs, and select the fastest
available compute device (GPU if present, otherwise CPU).


In [3]:
def set_seed(seed: int) -> None:
    '''Seed Python, NumPy, and PyTorch RNGs for reproducible results.

    Args:
        seed: The random seed to apply across all libraries.
    '''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    '''Select the best available compute device.

    Returns:
        A ``torch.device`` pointing to a CUDA GPU if available, otherwise the CPU.
    '''
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


set_seed(CONFIG.seed)
device = get_device()
print(f"Using device: {device}")


Using device: cpu


In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version PyTorch was built with:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
import sys
print(sys.executable)

ModuleNotFoundError: No module named 'torch'

## Data Loading

Instead of relying on torchvision's built-in `CelebA` dataset (which downloads
files from Google Drive and is prone to daily quota errors and, on Windows,
permission errors when creating its cache folder), this notebook loads
images directly from a **local folder of pre-downloaded CelebA images**
(e.g. the extracted `img_align_celeba` directory).

Since an autoencoder only needs images (no attribute labels), a lightweight
custom `Dataset` that reads `.jpg` files from disk is simpler and more
robust than the official loader. A single seeded split then carves out
non-overlapping train and validation subsets from the full image pool.

> **Set `Config.data_root` to the folder containing your CelebA `.jpg`
> images** before running this cell.


In [4]:
def build_transforms(image_size: int) -> transforms.Compose:
    '''Build the preprocessing pipeline applied to every CelebA image.

    Args:
        image_size: Target height/width for resizing and center-cropping.

    Returns:
        A composed torchvision transform.
    '''
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.CenterCrop((image_size, image_size)),
        transforms.ToTensor(),
    ])


class CelebAImageFolder(Dataset):
    '''Loads CelebA face images directly from a local folder of image files.

    Unlike ``torchvision.datasets.CelebA``, this does not require the
    official Google Drive download or attribute/partition annotation files —
    only the raw ``.jpg``/``.png`` images, which is all an autoencoder needs.

    Args:
        image_dir: Directory containing CelebA image files.
        transform: Preprocessing pipeline applied to each loaded image.
    '''

    _VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

    def __init__(self, image_dir: str, transform: transforms.Compose):
        self.image_dir = Path(image_dir)
        if not self.image_dir.is_dir():
            raise FileNotFoundError(
                f"CelebA image directory not found: {self.image_dir}\n"
                "Update Config.data_root to point to your local folder of "
                "CelebA .jpg images (e.g. the extracted 'img_align_celeba' folder)."
            )

        self.image_paths = sorted(
            path for path in self.image_dir.iterdir()
            if path.suffix.lower() in self._VALID_EXTENSIONS
        )
        if not self.image_paths:
            raise FileNotFoundError(f"No image files found in: {self.image_dir}")

        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        image = Image.open(self.image_paths[index]).convert("RGB")
        image = self.transform(image)
        return image, 0  # Dummy label kept only for API parity with the training loop.


def load_celeba_dataset(image_dir: str, transform: transforms.Compose) -> Dataset:
    '''Load all CelebA images from a local folder into a single dataset.

    Args:
        image_dir: Directory containing CelebA image files.
        transform: Preprocessing pipeline applied to each image.

    Returns:
        A ``CelebAImageFolder`` dataset over every image found in ``image_dir``.
    '''
    return CelebAImageFolder(image_dir, transform=transform)


def build_dataloaders(config: Config) -> Tuple[DataLoader, DataLoader]:
    '''Construct non-overlapping train and validation dataloaders for CelebA.

    A single seeded three-way split draws a training subset, a validation
    subset, and an unused remainder from the full local image pool, so the
    two subsets never share images.

    Args:
        config: Experiment configuration.

    Returns:
        A tuple of ``(train_loader, val_loader)``.

    Raises:
        ValueError: If more images are requested than are available.
    '''
    transform = build_transforms(config.image_size)
    full_dataset = load_celeba_dataset(config.data_root, transform)

    total_requested = config.train_subset_size + config.val_subset_size
    if total_requested > len(full_dataset):
        raise ValueError(
            f"Requested {total_requested} images (train + val) but only "
            f"{len(full_dataset)} images were found in {config.data_root}."
        )

    remainder_size = len(full_dataset) - total_requested
    generator = torch.Generator().manual_seed(config.seed)
    train_subset, val_subset, _ = random_split(
        full_dataset,
        [config.train_subset_size, config.val_subset_size, remainder_size],
        generator=generator,
    )

    train_loader = DataLoader(train_subset, batch_size=config.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_subset, batch_size=config.batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader


train_loader, val_loader = build_dataloaders(CONFIG)
print(f"Training batches: {len(train_loader)} | Validation batches: {len(val_loader)}")


Training batches: 79 | Validation batches: 8


## Model Architecture

The autoencoder is a symmetric encoder-decoder built from strided
convolutions and transposed convolutions:

- **Encoder:** three `Conv2d` layers progressively downsample the image
  (`64 → 32 → 16 → 8` spatial resolution) while increasing channel depth.
- **Decoder:** three `ConvTranspose2d` layers mirror the encoder to restore
  the original resolution, ending with a `Sigmoid` activation so outputs lie
  in the same `[0, 1]` range as the input tensors.


In [5]:
class ConvAutoencoder(nn.Module):
    '''A convolutional autoencoder for compressing and reconstructing face images.

    The encoder downsamples a ``(C, H, W)`` image through three strided
    convolutions, and the decoder mirrors this with transposed convolutions
    to reconstruct the original resolution.

    Args:
        in_channels: Number of input/output image channels.
        base_channels: Number of channels in the first encoder layer; deeper
            layers use multiples of this value.
    '''

    def __init__(self, in_channels: int = 3, base_channels: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Encode and reconstruct an input batch of images.

        Args:
            x: Input tensor of shape ``(N, C, H, W)``.

        Returns:
            Reconstructed tensor of the same shape as ``x``.
        '''
        latent = self.encoder(x)
        return self.decoder(latent)


model = ConvAutoencoder(in_channels=CONFIG.channels).to(device)
model


ConvAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): ReLU(inplace=True)
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): ConvTranspose2d(32, 3, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): Sigmoid()
  )
)

## Model Training

The model is trained to minimize pixel-wise **mean squared error** between
the reconstructed and original images using the Adam optimizer.

> **Improvement over the original script:** validation loss is now computed
> at the end of every epoch (in addition to training loss) so overfitting can
> be monitored directly from the loss curve, rather than only inspecting
> reconstructions at the very end.


In [6]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> float:
    '''Run a single training epoch over the given dataloader.

    Args:
        model: The autoencoder being trained.
        loader: Dataloader yielding ``(images, labels)`` batches.
        criterion: Reconstruction loss function.
        optimizer: Optimizer used to update model weights.
        device: Device to run computation on.

    Returns:
        The average training loss over all batches in the epoch.
    '''
    model.train()
    running_loss = 0.0
    for images, _ in loader:
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    '''Compute the average reconstruction loss on a validation/test loader.

    Args:
        model: The autoencoder being evaluated.
        loader: Dataloader yielding ``(images, labels)`` batches.
        criterion: Reconstruction loss function.
        device: Device to run computation on.

    Returns:
        The average loss over all batches in the loader.
    '''
    model.eval()
    running_loss = 0.0
    for images, _ in loader:
        images = images.to(device)
        outputs = model(images)
        loss = criterion(outputs, images)
        running_loss += loss.item()

    return running_loss / len(loader)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> Dict[str, List[float]]:
    '''Train the autoencoder for ``config.epochs`` epochs, tracking loss history.

    Args:
        model: The autoencoder to train.
        train_loader: Dataloader for the training subset.
        val_loader: Dataloader for the validation subset.
        config: Experiment configuration.
        device: Device to run computation on.

    Returns:
        A dictionary with ``"train_loss"`` and ``"val_loss"`` lists, one
        entry per epoch.
    '''
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    history: Dict[str, List[float]] = {"train_loss": [], "val_loss": []}

    for epoch in range(config.epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch [{epoch + 1}/{config.epochs}] "
            f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
        )

    return history


history = train_model(model, train_loader, val_loader, CONFIG, device)


Epoch [1/100] | Train Loss: 0.0848 | Val Loss: 0.0513
Epoch [2/100] | Train Loss: 0.0310 | Val Loss: 0.0252
Epoch [3/100] | Train Loss: 0.0222 | Val Loss: 0.0193
Epoch [4/100] | Train Loss: 0.0175 | Val Loss: 0.0159


KeyboardInterrupt: 

## Evaluation

The plot below shows how training and validation loss evolve over epochs,
making it easy to spot underfitting, overfitting, or a healthy converging
trend.


In [ ]:
def plot_loss_history(history: Dict[str, List[float]]) -> None:
    '''Plot training and validation loss curves over epochs.

    Args:
        history: Dictionary with ``"train_loss"`` and ``"val_loss"`` lists.
    '''
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Autoencoder Training Progress")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_loss_history(history)


## Visualization

Finally, we qualitatively assess reconstruction quality by comparing original
validation images to their autoencoder reconstructions side by side.


In [ ]:
@torch.no_grad()
def visualize_reconstructions(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    n_images: int = 4,
) -> None:
    '''Display original images alongside their autoencoder reconstructions.

    Args:
        model: Trained autoencoder.
        loader: Dataloader to draw a sample batch from.
        device: Device to run inference on.
        n_images: Number of image pairs to display.
    '''
    model.eval()
    images, _ = next(iter(loader))
    images = images.to(device)
    reconstructions = model(images)

    plt.figure(figsize=(2 * n_images, 4))
    for i in range(n_images):
        plt.subplot(2, n_images, i + 1)
        plt.imshow(images[i].cpu().permute(1, 2, 0))
        plt.title("Original")
        plt.axis("off")

        plt.subplot(2, n_images, i + 1 + n_images)
        plt.imshow(reconstructions[i].cpu().permute(1, 2, 0))
        plt.title("Reconstructed")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


visualize_reconstructions(model, val_loader, device, n_images=4)


## Results

After training, the loss curve should show a steadily decreasing training
loss that tracks closely with validation loss, indicating the autoencoder is
learning a generalizable compressed representation of faces rather than
memorizing the training subset. The reconstructed images should preserve
overall facial structure, pose, and lighting, with some loss of fine detail
(a natural side effect of compressing to a low-dimensional latent space).


## Conclusion

This notebook demonstrated an end-to-end, GPU-accelerated pipeline for
training a convolutional autoencoder on a subset of the CelebA dataset,
covering data loading, model design, training with validation tracking, and
qualitative evaluation through reconstruction visualization.


## Future Improvements

- Train on the full CelebA dataset rather than a fixed subset, given
  sufficient compute.
- Add a learning-rate scheduler and early stopping based on validation loss.
- Replace the vanilla autoencoder with a **Variational Autoencoder (VAE)** to
  enable structured sampling from the latent space.
- Add perceptual loss (e.g., VGG-based) in addition to MSE for sharper
  reconstructions.
- Log metrics and sample reconstructions to TensorBoard or Weights & Biases
  for richer experiment tracking.
- Add automated tests for the data pipeline and model shapes.


## References

- Liu, Z., Luo, P., Wang, X., & Tang, X. (2015). *Deep Learning Face
  Attributes in the Wild.* Proceedings of the IEEE International Conference
  on Computer Vision (ICCV).
- [CelebA Dataset homepage](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html)
- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)
